## Learning goals

By the end of this notebook, you should be able to:

1. Connect to MongoDB from Python with `PyMongo`.
2. Organize connection logic inside a class.
3. Create reusable methods for select, insert, update, and delete.
4. Read an external raw CSV dataset.
5. Create a new collection and ingest data into it.
6. Run a few small checks to confirm the ingestion worked.

## Dataset choice for this practice

We will use the UCI Student Performance dataset already stored in:

- `../../raw/uci-student-performance/dataset/student-mat.csv`
- `../../raw/uci-student-performance/dataset/student-por.csv`

These files use `;` as the separator.

The new collection to create in MongoDB will be:

- `student_performance`

## Imports

Complete the imports needed for:

- file paths
- JSON handling
- pandas
- MongoDB client

In [83]:
from pathlib import Path
import json

import pandas as pd
from pymongo import MongoClient
from pymongo.errors import OperationFailure, ServerSelectionTimeoutError

## Connection settings

Use the same MongoDB service from the previous practical class.

Credentials source for this notebook:
- `prj/docker-compose.yml`
- root user: `labuser`
- root password: `labpass`
- authentication database: `admin`
- recommended host in this notebook: `host.docker.internal`

If the connection fails with `Authentication failed`, first test the container manually with:
- `docker exec -it lab-mongo mongosh -u labuser -p labpass --authenticationDatabase admin`

If that command also fails, the running volume was probably initialized earlier with different credentials.

Important local environment note:
- if you already have another local `mongod` running on port `27017`, `localhost` may connect to that service instead of the class container
- for this reason, prefer `host.docker.internal` as the MongoDB host in the notebook configuration

Database to reuse:
- `university_mongo`

New collection to create for this class:
- `student_performance`

In [86]:
# TODO: define the connection settings
# Use the credentials declared in prj/docker-compose.yml
# Prefer host.docker.internal instead of localhost to avoid a local mongod conflict
# Remember that authentication is performed against the admin database
# Example variables to create:
MONGO_CONFIG = {
    "user": "labuser",
    "password": "labpass",
    "host": "localhost",
    "port": 27017,
    "database": "university_mongo"
}

RAW_DIR = Path("../raw/uci-student-performance/dataset")
COLLECTION_NAME = "student_performance"


## Documentation references:

- MongoDB PyMongo docs: [MongoClient](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.13/connect/mongoclient/)
- MongoDB PyMongo docs: [Find](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/find/)
- MongoDB PyMongo docs: [Insert](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/insert/)
- MongoDB PyMongo docs: [Update](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.10/crud/update/)
- MongoDB PyMongo docs: [Delete](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.10/crud/delete/)

- UCI dataset page: Student Performance: https://archive.ics.uci.edu/dataset/320/student+performance

## Step 1 - Create a database class

Model your solution with the same idea used in previous database exercises:

- store configuration in the constructor
- create a `connect()` method
- create a `close()` method
- provide a method to retrieve a collection

This class should focus only on connection management.

In [90]:
class MongoDatabase:
    """Class responsible for MongoDB connection management."""

    def __init__(self, db_config):
        # store configuration fields
        self.host = db_config["host"]
        self.port = db_config["port"]
        self.user = db_config["user"]
        self.password = db_config["password"]
        self.database_name = db_config["database"]

        self.client = None
        self.db = None

    def connect(self):
        # create client only if it does not exist yet
        if self.client is None:

            uri = (
                f"mongodb://{self.user}:{self.password}"
                f"@{self.host}:{self.port}/"
                f"?authSource=admin"
            )

            try:
                self.client = MongoClient(
                    uri,
                    serverSelectionTimeoutMS=3000
                )

                # select database
                self.db = self.client[self.database_name]

                # test connection
                self.client.admin.command("ping")

                print("Connected to MongoDB successfully.")

            except OperationFailure:
                raise Exception(
                    "Authentication failed.\n"
                    "Check the credentials in prj/docker-compose.yml\n"
                    "and test manually with:\n"
                    "docker exec -it lab-mongo mongosh "
                    "-u labuser -p labpass "
                    "--authenticationDatabase admin"
                )

            except ServerSelectionTimeoutError:
                raise Exception(
                    "Could not connect to MongoDB server."
                )

        return self.db

    def is_connected(self):
        # return True if connected
        return self.client is not None

    def get_collection(self, collection_name):
        # return collection object
        return self.db[collection_name]

    def close(self):
        # close client and reset references
        if self.client is not None:
            self.client.close()
            self.client = None
            self.db = None

            print("MongoDB connection closed.")

## Step 2 - Create a CRUD executor class

This second class should focus on collection operations, not on connection setup.

Suggested responsibilities:

- select documents
- insert one document
- insert many documents for ingestion
- update one document
- delete one document
- count documents

In [93]:
class MongoExecutor:
    """Class responsible for CRUD operations on a MongoDB collection."""

    def __init__(self, database, collection_name):
        # keep reference to database object
        self.database = database

        # obtain collection from database object
        self.collection = self.database.get_collection(collection_name)

    def select(self, filter_query=None, projection=None, limit=10):
        # simple find() wrapper
        if filter_query is None:
            filter_query = {}

        cursor = self.collection.find(
            filter_query,
            projection
        ).limit(limit)

        return list(cursor)

    def insert_one(self, document):
        # insert single document
        result = self.collection.insert_one(document)

        return result.inserted_id

    def insert_many(self, documents):
        # insert many documents
        result = self.collection.insert_many(documents)

        return result.inserted_ids

    def update_one(self, filter_query, update_query):
        # update single document
        result = self.collection.update_one(
            filter_query,
            update_query
        )

        return result.modified_count

    def delete_one(self, filter_query):
        # delete single document
        result = self.collection.delete_one(filter_query)

        return result.deleted_count

    def count_documents(self, filter_query=None):
        # count documents
        if filter_query is None:
            filter_query = {}

        return self.collection.count_documents(filter_query)


## Step 3 - Instantiate and test the connection

At this point, you should:

1. instantiate the `MongoDatabase`
2. connect to the database
3. instantiate the `MongoExecutor` for the new collection
4. confirm that the connection is working

Recommended check:
- call `db.connect()`
- if it fails, verify the same credentials with `mongosh` inside the container first

In [96]:
# instantiate database object
db = MongoDatabase(MONGO_CONFIG)

# connect to MongoDB
db.connect()

# instantiate executor for the collection
executor = MongoExecutor(db, COLLECTION_NAME)

# confirm connection is working
count = executor.count_documents()

print(f"Connection successful. Collection contains {count} documents.")

Connected to MongoDB successfully.
Connection successful. Collection contains 2098 documents.


### Test CRUD methods with small document

In [99]:
demo_document = {
    "external_student_id": "DEMO-00001",
    "source_dataset": "manual_demo",
    "source_file": "manual",
    "school": "GP",
    "sex": "F",
    "age": 17,
    "studytime": 2,
    "failures": 0,
    "absences": 1,
    "G1": 12,
    "G2": 13,
    "G3": 14,
    "internet": "yes",
    "higher": "yes"
}

# uncomment the following lines to test the CRUD methods with a small document
executor.insert_one(demo_document)
executor.update_one({"external_student_id": "DEMO-00001"}, {"$set": {"G3": 15}})
executor.select({"external_student_id": "DEMO-00001"}, {"_id": 0}, limit=1)


[{'external_student_id': 'DEMO-00001',
  'source_dataset': 'manual_demo',
  'source_file': 'manual',
  'school': 'GP',
  'sex': 'F',
  'age': 17,
  'studytime': 2,
  'failures': 0,
  'absences': 1,
  'G1': 12,
  'G2': 13,
  'G3': 15,
  'internet': 'yes',
  'higher': 'yes'}]

In [101]:
# delete the demo document by id after testing
executor.delete_one({"external_student_id": "DEMO-00001"})

1

## Step 4 - Read the raw CSV files

Read both CSV files from `RAW_DIR`.

Remember:
- the separator is `;`
- we are still not doing EDA
- just load and inspect a few rows

In [104]:
# create paths for CSV files
mat_path = RAW_DIR / "student-mat.csv"
por_path = RAW_DIR / "student-por.csv"

# load datasets (separator is ';')
df_mat = pd.read_csv(mat_path, sep=";")
df_por = pd.read_csv(por_path, sep=";")

# display a small sample
print("Math dataset sample:")
display(df_mat.head())

print("Portuguese dataset sample:")
display(df_por.head())


Math dataset sample:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


Portuguese dataset sample:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


## Step 5 - Map a raw row into a MongoDB document

Instead of inserting the raw CSV row blindly, create a mapping function.

Suggested output fields:

- `external_student_id`
- `source_dataset`
- `source_file`
- `school`
- `sex`
- `age`
- `studytime`
- `failures`
- `absences`
- `G1`, `G2`, `G3`
- `internet`
- `higher`


In [107]:
def map_external_record(record, dataset_name, row_number):
    """Transform one CSV row into one MongoDB document."""

    # TODO: build a stable document structure for the new collection
    # TODO: use a generated external_student_id such as UCI-MAT-00001
    # TODO: keep the source metadata in the document
    # TODO: cast numeric fields when appropriate
    
    external_id = f"UCI-{dataset_name.upper()}-{row_number:05d}"

    document = {
        "external_student_id": external_id,
        "source_dataset": dataset_name,
        "source_file": f"{dataset_name}.csv",

        "school": record["school"],
        "sex": record["sex"],
        "age": int(record["age"]),
        "studytime": int(record["studytime"]),
        "failures": int(record["failures"]),
        "absences": int(record["absences"]),

        # grades
        "G1": int(record["G1"]),
        "G2": int(record["G2"]),
        "G3": int(record["G3"]),

        # categorical features
        "internet": record["internet"],
        "higher": record["higher"]
    }

    return document



## Step 6 - Create the ingestion function

The ingestion function should:

1. read one CSV file
2. optionally limit the number of rows
3. map each row to a document
4. insert the documents into `student_performance_raw`
5. return how many documents were inserted

In [110]:
def ingest_data(executor, csv_path, limit=None):
    """Read a CSV file and insert its mapped rows into MongoDB."""

    # TODO: load the CSV with pandas
    # TODO: apply the optional limit
    # TODO: convert rows to mapped documents
    # TODO: insert documents into MongoDB
    # TODO: return the number of inserted documents

    df = pd.read_csv(csv_path, sep=";")

    if limit is not None:
        df = df.head(limit)

    dataset_name = csv_path.stem.split("-")[1]  # "mat" or "por"

    documents = [
        map_external_record(row, dataset_name, i)
        for i, row in df.iterrows()
    ]

    inserted_ids = executor.insert_many(documents)

    return len(inserted_ids)


## Step 7 - Small validation checks

Do not perform EDA yet.

Only validate that the pipeline is working. Examples:

- total documents in the new collection
- total documents by source file
- one sample document after ingestion
- one manual insert/update/delete cycle if needed

In [113]:
# ingest a small batch first, for example 5 or 10 rows
inserted_small = ingest_data(executor, mat_path, limit=10)
print("Inserted (small batch):", inserted_small)
# validate the collection count
total_docs = executor.count_documents()
print("Total documents in collection:", total_docs)
# inspect one sample document
sample = executor.select(limit=1)
print(sample)
# if everything looks good, ingest the full file(s)


Inserted (small batch): 10
Total documents in collection: 2108
[{'_id': ObjectId('6a0332253aaa37b4c1293f09'), 'external_student_id': 'UCI-MAT-00000', 'source_dataset': 'mat', 'source_file': 'mat.csv', 'school': 'GP', 'sex': 'F', 'age': 18, 'studytime': 2, 'failures': 0, 'absences': 6, 'G1': 5, 'G2': 6, 'G3': 6, 'internet': 'no', 'higher': 'yes'}]


In [115]:
ingest_data(executor, mat_path)
ingest_data(executor, por_path)


649

## Suggested class tasks

1. Finish the `MongoDatabase` class.
2. Finish the `MongoExecutor` class.
3. Connect to `university_mongo`.
4. Create and use the new collection `student_performance_raw`.
5. Load `student-mat.csv`.
6. Implement `map_external_record()`.
7. Implement `ingest_data()`.
8. Validate counts and inspect one ingested document.
9. Repeat for `student-por.csv` if time allows.

This notebook should end with a working ingestion pipeline, not with EDA.